# 01 — Giriş Problemi: Bulut Sunucusu

**Öğrenme hedefleri:**
1. Decision variable, objective function ve constraint kavramlarını somut bir problem üzerinden anlamak
2. Feasible region'ı grafikle görmek
3. LP (sürekli) ile ILP (tam sayı) arasındaki farkı gözlemlemek
4. Infeasible ve unbounded durumları tanımak

**Tahmini süre:** ~30 dakika

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import numpy as np
import matplotlib.pyplot as plt
from utils.plotting import plot_feasible_region_cloud

---
## Problem Tanımı

Bir şirket iki tip bulut sunucusu kiralayabilir.  
Toplamda en az **30 vCPU** ve **90 GB RAM** gerekiyor.  
**Hedef:** Saatlik maliyeti en aza indirmek için her tipten kaç sunucu kiralanmalı?

| Tip | vCPU | RAM (GB) | Saatlik maliyet (\$) |
|-----|------|----------|----------------------|
| A   | 4    | 8        | 10                   |
| B   | 8    | 32       | 25                   |

---
## Kavramlar: ILP'nin Yapı Taşları

### Decision Variables (Karar Değişkenleri)
Ne kadar/kaç tane alacağımıza karar veriyoruz:
- `a` = kiralanacak Tip-A sunucu sayısı  
- `b` = kiralanacak Tip-B sunucu sayısı

### Objective Function (Amaç Fonksiyonu)
Neyi optimize ediyoruz?  
`minimize  10·a + 25·b`

### Constraints (Kısıtlar)
Hangi koşullar sağlanmalı?  
- `4a + 8b ≥ 30`   (vCPU ihtiyacı)  
- `8a + 32b ≥ 90`  (RAM ihtiyacı)  
- `a, b ≥ 0`       (negatif sunucu olmaz)

---
## Adım 1 — LP Çözümü (sürekli değişkenler)

Önce `a` ve `b`'yi **sürekli** (kesirli) kabul edelim.

In [ ]:
prob_lp = pulp.LpProblem("cloud_server_LP", pulp.LpMinimize)

a = pulp.LpVariable("a", lowBound=0, cat="Continuous")
b = pulp.LpVariable("b", lowBound=0, cat="Continuous")

# Objective
prob_lp += 10 * a + 25 * b

# Constraints
prob_lp += 4 * a + 8 * b >= 30,   "vcpu"
prob_lp += 8 * a + 32 * b >= 90,  "ram"

prob_lp.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Durum   : {pulp.LpStatus[prob_lp.status]}")
print(f"a (LP)  : {pulp.value(a):.4f}")
print(f"b (LP)  : {pulp.value(b):.4f}")
print(f"Maliyet : ${pulp.value(prob_lp.objective):.4f}")

> **Soru:** 3.75 sunucu kiralayabilir miyiz? Bu sonuç pratikte ne anlama geliyor?

---
## Adım 2 — Feasible Region Grafiği

In [ ]:
a_lp_val = pulp.value(a)
b_lp_val = pulp.value(b)

fig = plot_feasible_region_cloud(a_lp=a_lp_val, b_lp=b_lp_val)
plt.show()

---
## Adım 3 — ILP Çözümü (tam sayı kısıtı)

Şimdi `a` ve `b`'yi **Integer** (tam sayı) olarak tanımlayalım.

In [ ]:
prob_ilp = pulp.LpProblem("cloud_server_ILP", pulp.LpMinimize)

a_i = pulp.LpVariable("a", lowBound=0, cat="Integer")
b_i = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_ilp += 10 * a_i + 25 * b_i
prob_ilp += 4 * a_i + 8 * b_i >= 30,  "vcpu"
prob_ilp += 8 * a_i + 32 * b_i >= 90, "ram"

prob_ilp.solve(pulp.PULP_CBC_CMD(msg=0))

a_ilp_val = int(pulp.value(a_i))
b_ilp_val = int(pulp.value(b_i))

print(f"Durum   : {pulp.LpStatus[prob_ilp.status]}")
print(f"a (ILP) : {a_ilp_val}")
print(f"b (ILP) : {b_ilp_val}")
print(f"Maliyet : ${pulp.value(prob_ilp.objective):.2f}")

---
## Adım 4 — Grafik: LP vs ILP Karşılaştırması

In [ ]:
fig = plot_feasible_region_cloud(
    a_lp=a_lp_val, b_lp=b_lp_val,
    a_ilp=a_ilp_val, b_ilp=b_ilp_val
)
plt.show()

print(f"LP  maliyet: ${10*a_lp_val + 25*b_lp_val:.3f}  (alt sınır)")
print(f"ILP maliyet: ${10*a_ilp_val + 25*b_ilp_val:.2f}")
print(f"Fark       : ${10*a_ilp_val + 25*b_ilp_val - (10*a_lp_val + 25*b_lp_val):.3f}")

---
## Adım 5 — Aşağı Yuvarlama Neden Çalışmaz?

LP çözümünü aşağı yuvarlarsak (`a=3, b=1`) kısıtlar sağlanıyor mu?

In [ ]:
a_round, b_round = 3, 1

vcpu_ok = (4 * a_round + 8 * b_round >= 30)
ram_ok  = (8 * a_round + 32 * b_round >= 90)

print(f"a=3, b=1 için:")
print(f"  vCPU: 4*3 + 8*1 = {4*a_round + 8*b_round}  >= 30 ?  {'✅' if vcpu_ok else '❌'}")
print(f"  RAM : 8*3 + 32*1 = {8*a_round + 32*b_round} >= 90 ?  {'✅' if ram_ok else '❌'}")
print()
print("=> Aşağı yuvarlama infeasible bir çözüm üretebilir — solver bu yüzden gerekli!")

---
## 🔧 Değiştir-Gözlemle Görevleri

### Görev 1 — Ek Kısıt: En Fazla 1 Tip-B Sunucu

> **Tahmin:** "B tipi sunucudan en fazla 1 tane kiralanabilir" kısıtını eklersen maliyet nasıl değişir?  
> *(Cevabı yazın, sonra kodu çalıştırın)*

In [ ]:
prob_g1 = pulp.LpProblem("cloud_g1", pulp.LpMinimize)
ag1 = pulp.LpVariable("a", lowBound=0, cat="Integer")
bg1 = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_g1 += 10 * ag1 + 25 * bg1
prob_g1 += 4 * ag1 + 8 * bg1 >= 30
prob_g1 += 8 * ag1 + 32 * bg1 >= 90
prob_g1 += bg1 <= 1  # YENİ KISIT

prob_g1.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum   : {pulp.LpStatus[prob_g1.status]}")
print(f"a={int(pulp.value(ag1))}, b={int(pulp.value(bg1))}, maliyet=${pulp.value(prob_g1.objective):.2f}")

> **Eğitmen notu:** Beklenen sonuç `a=8, b=1`, maliyet `$105`. Tek B sunucusundan gelen 32 GB RAM yeterli değil, fazladan A sunucuları devreye giriyor.

### Görev 2 — Bütçe Kısıtı: Saatlik En Fazla \$80

> **Tahmin:** Bütçeyi \$80 ile sınırlarsak ne olur?  
> *(LP'nin optimal değeri bile 84.375\$ idi — bunu hatırlayın)*

In [ ]:
prob_g2 = pulp.LpProblem("cloud_g2", pulp.LpMinimize)
ag2 = pulp.LpVariable("a", lowBound=0, cat="Integer")
bg2 = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_g2 += 10 * ag2 + 25 * bg2
prob_g2 += 4 * ag2 + 8 * bg2 >= 30
prob_g2 += 8 * ag2 + 32 * bg2 >= 90
prob_g2 += 10 * ag2 + 25 * bg2 <= 80  # BÜTÇE KISITI

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum: {pulp.LpStatus[prob_g2.status]}")

> **Eğitmen notu:** Durum **Infeasible**. LP minimumu bile \$84.375 olduğundan, \$80 bütçesiyle hem vCPU hem RAM ihtiyacını karşılamak mümkün değil.

### Görev 3 — LpMaximize: Ne Olur?

> **Tahmin:** Minimize yerine maximize yaparsak?

In [ ]:
prob_g3 = pulp.LpProblem("cloud_g3", pulp.LpMaximize)  # MAX!
ag3 = pulp.LpVariable("a", lowBound=0, cat="Continuous")
bg3 = pulp.LpVariable("b", lowBound=0, cat="Continuous")

prob_g3 += 10 * ag3 + 25 * bg3
prob_g3 += 4 * ag3 + 8 * bg3 >= 30
prob_g3 += 8 * ag3 + 32 * bg3 >= 90

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum: {pulp.LpStatus[prob_g3.status]}")

> **Eğitmen notu:** Durum **Unbounded**. '>=' kısıtlarıyla maximize yaparken `a` ve `b` sonsuz artabilir. Bu durum genellikle modelleme hatasına işaret eder.

### Görev 4 — Üçüncü Sunucu Tipi (Serbest)

> Kendi seçtiğin değerlerle bir Tip-C sunucu ekle.  
> Grafiği artık 2 boyutta çizmek mümkün değil — solver'ın gücü burada ortaya çıkıyor!

In [ ]:
# TODO: Tip-C için vCPU, RAM ve maliyet değerlerini belirle
# vcpu_c = ?
# ram_c  = ?
# cost_c = ?

# prob_g4 = pulp.LpProblem("cloud_g4", pulp.LpMinimize)
# a4 = pulp.LpVariable("a", lowBound=0, cat="Integer")
# b4 = pulp.LpVariable("b", lowBound=0, cat="Integer")
# c4 = pulp.LpVariable("c", lowBound=0, cat="Integer")
# prob_g4 += 10*a4 + 25*b4 + cost_c*c4
# prob_g4 += 4*a4 + 8*b4 + vcpu_c*c4 >= 30
# prob_g4 += 8*a4 + 32*b4 + ram_c*c4  >= 90
# prob_g4.solve(pulp.PULP_CBC_CMD(msg=0))
# print(f"Durum: {pulp.LpStatus[prob_g4.status]}")
# print(f"a={int(pulp.value(a4))}, b={int(pulp.value(b4))}, c={int(pulp.value(c4))}")
# print(f"Maliyet: ${pulp.value(prob_g4.objective):.2f}")
print("Yukarıdaki yorumları kaldırıp değerleri doldurun!")